In [1]:
import pandas as pd
import numpy as np
import re
import nltk
import pickle
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Download required NLTK data
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')

# Load dataset
df = pd.read_csv(r"C:\Users\Matthew .K. Maunga\OneDrive\Desktop\MSU project\MSU_Lost_Found_ML_System\data\raw\descriptions.csv")

print(f" Dataset loaded: {len(df)} records")
print(f"Columns: {list(df.columns)}")

[nltk_data] Downloading package stopwords to C:\Users\Matthew .K.
[nltk_data]     Maunga\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\Matthew .K.
[nltk_data]     Maunga\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to C:\Users\Matthew .K.
[nltk_data]     Maunga\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


 Dataset loaded: 172 records
Columns: ['item_id', 'item_name', 'description', 'category', 'status', 'date_reported', 'image_path', 'label']


In [2]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    # Lowercase
    text = text.lower()
    # Remove punctuation and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Tokenize
    tokens = text.split()
    # Remove stopwords and lemmatize
    tokens = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words]
    return ' '.join(tokens)

# Apply cleaning
df['cleaned_description'] = df['description'].apply(clean_text)

# Show comparison
print("Original vs Cleaned:\n")
for i in range(3):
    print(f"Original : {df['description'].iloc[i]}")
    print(f"Cleaned  : {df['cleaned_description'].iloc[i]}")
    print()

Original vs Cleaned:

Original : A small brown backpack with multiple pockets and a Puma logo on the front.
Cleaned  : small brown backpack multiple pocket puma logo front

Original : A large pink school backpack with side mesh pockets for water bottles.
Cleaned  : large pink school backpack side mesh pocket water bottle

Original : A Nike backpack in grey with a padded laptop sleeve inside.
Cleaned  : nike backpack grey padded laptop sleeve inside



In [3]:
# Load SBERT model
print("Loading SBERT model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print(" Model loaded!")

# Generate embeddings for all descriptions
print("\nGenerating embeddings for all 172 descriptions...")
embeddings = model.encode(
    df['cleaned_description'].tolist(),
    show_progress_bar=True
)

print(f"\n Embeddings generated!")
print(f"   Shape: {embeddings.shape}")
print(f"   Each description = {embeddings.shape[1]} dimensional vector")

Loading SBERT model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

 Model loaded!

Generating embeddings for all 172 descriptions...


Batches:   0%|          | 0/6 [00:00<?, ?it/s]


 Embeddings generated!
   Shape: (172, 384)
   Each description = 384 dimensional vector


In [4]:
# Save embeddings to pickle file
PROCESSED_PATH = r"C:\Users\Matthew .K. Maunga\OneDrive\Desktop\MSU project\MSU_Lost_Found_ML_System\data\processed"

# Save embeddings
with open(f"{PROCESSED_PATH}\\nlp_features.pkl", "wb") as f:
    pickle.dump({
        'embeddings': embeddings,
        'item_ids': df['item_id'].tolist(),
        'categories': df['category'].tolist(),
        'status': df['status'].tolist(),
        'descriptions': df['cleaned_description'].tolist()
    }, f)

print(" NLP features saved to data/processed/nlp_features.pkl")

# Test similarity between a lost and found item
lost_items  = df[df['status'] == 'lost'].reset_index(drop=True)
found_items = df[df['status'] == 'found'].reset_index(drop=True)

lost_embeddings  = embeddings[df['status'] == 'lost']
found_embeddings = embeddings[df['status'] == 'found']

# Compare first lost item against all found items
scores = cosine_similarity([lost_embeddings[0]], found_embeddings)[0]
best_match_idx = scores.argmax()

print(f"\n--- Test Match ---")
print(f"Lost item    : {lost_items['description'].iloc[0]}")
print(f"Best match   : {found_items['description'].iloc[best_match_idx]}")
print(f"NLP Score    : {scores[best_match_idx]:.4f}")
print(f"Category lost: {lost_items['category'].iloc[0]}")
print(f"Category found:{found_items['category'].iloc[best_match_idx]}")

 NLP features saved to data/processed/nlp_features.pkl

--- Test Match ---
Lost item    : A small brown backpack with multiple pockets and a Puma logo on the front.
Best match   : A white ceramic cup with a Puma logo printed on the side.
NLP Score    : 0.5809
Category lost: backpack
Category found:cup
